<a href="https://colab.research.google.com/github/Flottohs/Phase-1/blob/main/my_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math

class Value:
    def __init__(self, data, _children=(), label='',_op=''):
        self.data = data
        self.grad = 0.0  # Derivative of the final output with respect to this value

        #Internal variables for the graph
        self._backward = lambda: None  # The "recipe" for the chain rule
        self._prev = set(_children)    # Pointers to the values that created this one
        self._op = _op                 # The operation (for debugging if needed)
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad}, label='{self.label}')"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # Addition Rule: The gradient simply flows back unchanged
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # Product Rule: dL/dx = dz/dx * dL / dz , dL/dy = dz/dy * dL/dz


            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward

        return out

    def backward(self):
        # 1.Topological Sort: Orders nodes from beginning to end
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # 2. Base case: The gradient of the end result with respect to itself is 1
        self.grad = 1.0

        # 3. Go through the graph in reverse (end to beginning)
        for node in reversed(topo):
            node._backward()




### Example: A Single Neuron Trace
#Let's calculate $out = ReLU(w \cdot x + b)$ using specific values.

In [ ]:



# 1. Setup the inputs
w = Value(2.0, label='weight')
x = Value(-3.0, label='input')
b = Value(10.0, label='bias')

#2. The Forward Pass
# (2.0 * -3.0) + 10.0 = 4.0. Then ReLU(4.0) = 4.0
wx = w * x; wx.label = 'w*x'
z = wx + b; z.label = 'z'
out = z.relu(); out.label = 'out'

# 3. The Backward Pass
out.backward()

# 4. View the Results
print(f"Final Output: {out.data}")
print(f"Gradient of Weight (w): {w.grad}")
print(f"Gradient of Input (x) : {x.grad}")
print(f"Gradient of Bias (b)  : {b.grad}")

Final Output: 4.0
Gradient of Weight (w): -3.0
Gradient of Input (x) : 2.0
Gradient of Bias (b)  : 1.0


#Be careful with the distinction of finding the affect of a weight on the loss vs the input on the loss

input on the loss is multivariate chain rule
weight on the loss is standard chain rule